In [1]:
"""
03_train_valid_test_split.ipynb

Purpose:
Create leakage-safe stratified train/validation/test splits
from the processed Telstra-style training dataset.

What this notebook does:
1. Load processed training data
2. Separate ID, trace columns, target, and model features
3. Create stratified train/validation/test splits
4. Save the split datasets used by later notebooks
5. Save only small verification reports for viva

Outputs saved to:
    ../data/splits/
    ../reports/splits/
"""

# 1. Imports and Environment Setup

# sys is used only to print Python version
import sys

# Path helps create file paths cleanly
from pathlib import Path

# numpy is used for reproducibility
import numpy as np

# pandas is used for reading and saving tabular data
import pandas as pd

# train_test_split is used to make stratified splits
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except Exception:
    display = None

# Show more columns when printing dataframes
pd.set_option("display.max_columns", None)

# Make dataframe output wider in notebook
pd.set_option("display.width", 180)

# Fix random seed so the split stays reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Print versions
print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


# 2. Paths

# Assume notebook runs from /notebooks
PROJECT_ROOT = Path.cwd().parent

# Input: processed training data from preprocessing notebook
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

# Output: train/valid/test split files
SPLITS_PATH = PROJECT_ROOT / "data" / "splits"

# Small report files for verification
REPORTS_PATH = PROJECT_ROOT / "reports" / "splits"

# Create output folders if they do not already exist
SPLITS_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("PROCESSED_PATH :", PROCESSED_PATH)
print("SPLITS_PATH    :", SPLITS_PATH)
print("REPORTS_PATH   :", REPORTS_PATH)


# 3. Helper Functions

def show_df(df: pd.DataFrame, n: int = 5) -> None:
    """
    Show the first few rows of a dataframe.
    """
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n))


def class_distribution(y: pd.Series, split_name: str) -> pd.DataFrame:
    """
    Create a small table showing:
    - class label
    - count
    - proportion

    This helps check whether stratification worked correctly.
    """
    counts = y.value_counts().sort_index()
    props = y.value_counts(normalize=True).sort_index().round(4)

    return pd.DataFrame(
        {
            "split": split_name,
            "class": counts.index,
            "count": counts.values,
            "proportion": props.values,
        }
    )


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Save dataframe as CSV.
    """
    df.to_csv(path, index=index)


# 4. Load Processed Training Data

# Load the processed training table from notebook 02
train_processed = pd.read_csv(PROCESSED_PATH / "train_processed.csv")

print("\nLoaded train_processed shape:", train_processed.shape)
show_df(train_processed.iloc[:, :20])


# 5. Define Columns

print("\nDEFINING COLUMNS")

# id is used to track rows and check leakage
ID_COL = "id"

# location is not a model target, but useful as metadata for tracing cases
TRACE_COLS = ["location"]

# This is the target label we want to predict
TARGET_COL = "fault_severity"

# These columns should not be part of model features
non_feature_cols = [ID_COL] + TRACE_COLS + [TARGET_COL]

# Everything else becomes a model feature
feature_cols = [c for c in train_processed.columns if c not in non_feature_cols]

print("ID column         :", ID_COL)
print("Trace columns     :", TRACE_COLS)
print("Target column     :", TARGET_COL)
print("Number of features:", len(feature_cols))

# Safety checks
if TARGET_COL not in train_processed.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found.")

if ID_COL not in train_processed.columns:
    raise ValueError(f"ID column '{ID_COL}' not found.")

if len(feature_cols) == 0:
    raise ValueError("No feature columns found after excluding id/location/target.")


# 6. Build Full X / y / metadata

print("\nBUILDING X / y / METADATA")

# X contains only model input features
X = train_processed[feature_cols].copy()

# y contains the target label
y = train_processed[TARGET_COL].copy()

# meta keeps useful trace information separate from features
meta = train_processed[[ID_COL] + TRACE_COLS].copy()

print("X shape   :", X.shape)
print("y shape   :", y.shape)
print("meta shape:", meta.shape)

# Check class distribution before splitting
print("\nTarget distribution in full processed training set:")
full_dist = class_distribution(y, "full")
print(full_dist)


# 7. First Split: Development / Final Test

print("\nSPLIT 1: DEVELOPMENT / FINAL TEST")

# First split:
# - 80% goes to development set
# - 20% becomes final test set
# Stratify by y so class balance is preserved
X_dev, X_test, y_dev, y_test, meta_dev, meta_test = train_test_split(
    X,
    y,
    meta,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("X_dev shape :", X_dev.shape)
print("X_test shape:", X_test.shape)
print("y_dev shape :", y_dev.shape)
print("y_test shape:", y_test.shape)

# Distribution check after first split
dev_dist = class_distribution(y_dev, "dev")
test_dist = class_distribution(y_test, "test")

print("\nDevelopment set distribution:")
print(dev_dist)

print("\nFinal test set distribution:")
print(test_dist)


# 8. Second Split: Train / Validation

print("\nSPLIT 2: TRAIN / VALIDATION")

# Second split happens only inside the development set
# - 80% of dev becomes train
# - 20% of dev becomes valid
# Since dev is already 80% of full data,
# this gives about 64% train and 16% valid overall
X_train, X_valid, y_train, y_valid, meta_train, meta_valid = train_test_split(
    X_dev,
    y_dev,
    meta_dev,
    test_size=0.20,
    stratify=y_dev,
    random_state=RANDOM_STATE,
)

print("X_train shape :", X_train.shape)
print("X_valid shape :", X_valid.shape)
print("X_test shape  :", X_test.shape)

print("y_train shape :", y_train.shape)
print("y_valid shape :", y_valid.shape)
print("y_test shape  :", y_test.shape)

# Distribution check after second split
train_dist = class_distribution(y_train, "train")
valid_dist = class_distribution(y_valid, "valid")

print("\nTrain distribution:")
print(train_dist)

print("\nValidation distribution:")
print(valid_dist)

print("\nTest distribution:")
print(test_dist)


# 9. Save Small Class Distribution Report

print("\nSAVING CLASS DISTRIBUTION REPORT")

# Combine all distributions into one small report
split_dist = pd.concat(
    [full_dist, train_dist, valid_dist, test_dist],
    axis=0,
    ignore_index=True,
)

save_csv(split_dist, REPORTS_PATH / "class_distribution_all_splits.csv", index=False)

print("Saved class distribution report.")


# 10. Leakage-Safety and Integrity Checks

print("\nLEAKAGE / INTEGRITY CHECKS")

# Make sure row counts still match
n_total = len(train_processed)
n_split_total = len(X_train) + len(X_valid) + len(X_test)

print("Original rows :", n_total)
print("Split rows sum:", n_split_total)

if n_total != n_split_total:
    raise ValueError("Row count mismatch after splitting.")

# Extract ids from each split
train_ids = set(meta_train[ID_COL])
valid_ids = set(meta_valid[ID_COL])
test_ids = set(meta_test[ID_COL])

# Check for overlap between splits
# There should be no shared ids across train, valid, and test
print("Train/Valid overlap:", len(train_ids & valid_ids))
print("Train/Test overlap :", len(train_ids & test_ids))
print("Valid/Test overlap :", len(valid_ids & test_ids))

if len(train_ids & valid_ids) > 0:
    raise ValueError("Leakage detected: train and valid share IDs.")
if len(train_ids & test_ids) > 0:
    raise ValueError("Leakage detected: train and test share IDs.")
if len(valid_ids & test_ids) > 0:
    raise ValueError("Leakage detected: valid and test share IDs.")

# Check feature alignment
# All three feature sets must have the same columns in the same order
if list(X_train.columns) != list(X_valid.columns):
    raise ValueError("Feature mismatch between train and valid.")
if list(X_train.columns) != list(X_test.columns):
    raise ValueError("Feature mismatch between train and test.")

# Check missing values
print("Missing values in X_train:", int(X_train.isna().sum().sum()))
print("Missing values in X_valid:", int(X_valid.isna().sum().sum()))
print("Missing values in X_test :", int(X_test.isna().sum().sum()))

if X_train.isna().sum().sum() != 0:
    raise ValueError("X_train contains missing values.")
if X_valid.isna().sum().sum() != 0:
    raise ValueError("X_valid contains missing values.")
if X_test.isna().sum().sum() != 0:
    raise ValueError("X_test contains missing values.")

print("Integrity checks passed.")


# 11. Save Split Datasets

print("\nSAVING SPLIT DATASETS")

# Save feature matrices
save_csv(X_train, SPLITS_PATH / "X_train.csv", index=False)
save_csv(X_valid, SPLITS_PATH / "X_valid.csv", index=False)
save_csv(X_test, SPLITS_PATH / "X_test.csv", index=False)

# Save target vectors
save_csv(pd.DataFrame({"fault_severity": y_train}), SPLITS_PATH / "y_train.csv", index=False)
save_csv(pd.DataFrame({"fault_severity": y_valid}), SPLITS_PATH / "y_valid.csv", index=False)
save_csv(pd.DataFrame({"fault_severity": y_test}), SPLITS_PATH / "y_test.csv", index=False)

# Save metadata tables
save_csv(meta_train.reset_index(drop=True), SPLITS_PATH / "meta_train.csv", index=False)
save_csv(meta_valid.reset_index(drop=True), SPLITS_PATH / "meta_valid.csv", index=False)
save_csv(meta_test.reset_index(drop=True), SPLITS_PATH / "meta_test.csv", index=False)

# Save final feature list
feature_list_df = pd.DataFrame({"feature_name": feature_cols})
save_csv(feature_list_df, SPLITS_PATH / "feature_columns.csv", index=False)

print("Saved split datasets to:", SPLITS_PATH)


# 12. Split Summary

print("\nSPLIT SUMMARY")

# Small summary table for viva
summary = pd.DataFrame(
    {
        "split": ["train", "valid", "test"],
        "rows": [len(X_train), len(X_valid), len(X_test)],
        "pct_of_total": [
            round(len(X_train) / n_total, 4),
            round(len(X_valid) / n_total, 4),
            round(len(X_test) / n_total, 4),
        ],
        "n_features": [X_train.shape[1], X_valid.shape[1], X_test.shape[1]],
    }
)

print(summary)
save_csv(summary, REPORTS_PATH / "split_summary.csv", index=False)

print("\nExpected approximate ratios:")
print("Train ≈ 64%")
print("Valid ≈ 16%")
print("Test  ≈ 20%")


# 13. Final Notes

print("\nTRAIN / VALID / TEST SPLITTING COMPLETE")
print("Data saved to   :", SPLITS_PATH)
print("Reports saved to:", REPORTS_PATH)

Python : 3.12.2
Pandas : 2.3.3
NumPy  : 2.3.5
PROJECT_ROOT   : /Users/hasheenadilmidesilva/Desktop/NetGuard
PROCESSED_PATH : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/processed
SPLITS_PATH    : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/splits
REPORTS_PATH   : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/splits

Loaded train_processed shape: (7381, 863)


,id,location,fault_severity,location_num,severity_type_1,severity_type_2,severity_type_3,severity_type_4,severity_type_5,event_type_1,event_type_10,event_type_11,event_type_12,event_type_13,event_type_14,event_type_15,event_type_17,event_type_18,event_type_19,event_type_2
0,14121,location 118,1,118,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,9320,location 91,0,91,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,14394,location 152,1,152,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,8218,location 931,1,931,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
4,14804,location 120,0,120,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0



DEFINING COLUMNS
ID column         : id
Trace columns     : ['location']
Target column     : fault_severity
Number of features: 860

BUILDING X / y / METADATA
X shape   : (7381, 860)
y shape   : (7381,)
meta shape: (7381, 2)

Target distribution in full processed training set:
  split  class  count  proportion
0  full      0   4784      0.6482
1  full      1   1871      0.2535
2  full      2    726      0.0984

SPLIT 1: DEVELOPMENT / FINAL TEST
X_dev shape : (5904, 860)
X_test shape: (1477, 860)
y_dev shape : (5904,)
y_test shape: (1477,)

Development set distribution:
  split  class  count  proportion
0   dev      0   3827      0.6482
1   dev      1   1496      0.2534
2   dev      2    581      0.0984

Final test set distribution:
  split  class  count  proportion
0  test      0    957      0.6479
1  test      1    375      0.2539
2  test      2    145      0.0982

SPLIT 2: TRAIN / VALIDATION
X_train shape : (4723, 860)
X_valid shape : (1181, 860)
X_test shape  : (1477, 860)
y_train 